In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.datasets import load_iris
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report
from tensorflow.keras.datasets import mnist
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Input, Dense
from tensorflow.keras.utils import to_categorical
plt.style.use("seaborn-v0_8-whitegrid")
np.random.seed(42)


In [ ]:
def sigmoid(z):
    return 1 / (1 + np.exp(-z))
def softmax(z):
    exp_z = np.exp(z - np.max(z, axis=1, keepdims=True))
    return exp_z / np.sum(exp_z, axis=1, keepdims=True)
def one_hot(y, num_classes):
    result = np.zeros((y.shape[0], num_classes))
    result[np.arange(y.shape[0]), y] = 1
    return result


In [ ]:
class NeuralNetwork:
    def __init__(self, input_size, hidden_size, output_size, learning_rate):
        self.lr = learning_rate
        self.W1 = np.random.randn(input_size, hidden_size) * 0.01
        self.b1 = np.zeros((1, hidden_size))
        self.W2 = np.random.randn(hidden_size, output_size) * 0.01
        self.b2 = np.zeros((1, output_size))
        self.losses = []
    def forward(self, X):
        self.z1 = np.dot(X, self.W1) + self.b1
        self.a1 = sigmoid(self.z1)
        self.z2 = np.dot(self.a1, self.W2) + self.b2
        self.a2 = sigmoid(self.z2)
        return self.a2
    def train(self, X, y, epochs):
        y = y.reshape(-1, 1)
        for epoch in range(epochs):
            y_pred = self.forward(X)
            loss = -np.mean(
                y * np.log(y_pred + 1e-8) +
                (1 - y) * np.log(1 - y_pred + 1e-8)
            )
            self.losses.append(loss)
            m = X.shape[0]
            dz2 = y_pred - y
            dW2 = np.dot(self.a1.T, dz2) / m
            db2 = np.sum(dz2, axis=0, keepdims=True) / m
            dz1 = np.dot(dz2, self.W2.T) * self.a1 * (1 - self.a1)
            dW1 = np.dot(X.T, dz1) / m
            db1 = np.sum(dz1, axis=0, keepdims=True) / m
            self.W2 -= self.lr * dW2
            self.b2 -= self.lr * db2
            self.W1 -= self.lr * dW1
            self.b1 -= self.lr * db1
            if epoch % 500 == 0:
                print("Epoch:", epoch, "Loss:", loss)
    def predict(self, X):
        return (self.forward(X) >= 0.5).astype(int)


In [ ]:
class NeuralNetworkMNIST:
    def __init__(self, input_size, hidden_size, output_size, learning_rate):
        self.lr = learning_rate
        self.W1 = np.random.randn(input_size, hidden_size) * 0.01
        self.b1 = np.zeros((1, hidden_size))
        self.W2 = np.random.randn(hidden_size, output_size) * 0.01
        self.b2 = np.zeros((1, output_size))
        self.losses = []
    def forward(self, X):
        self.z1 = np.dot(X, self.W1) + self.b1
        self.a1 = sigmoid(self.z1)
        self.z2 = np.dot(self.a1, self.W2) + self.b2
        self.a2 = softmax(self.z2)
        return self.a2
    def train(self, X, y, epochs, batch_size=128):
        n = X.shape[0]
        for epoch in range(epochs):
            indices = np.random.permutation(n)
            X_shuffled = X[indices]
            y_shuffled = y[indices]
            for start in range(0, n, batch_size):
                end = start + batch_size
                X_batch = X_shuffled[start:end]
                y_batch = y_shuffled[start:end]
                y_pred = self.forward(X_batch)
                m = X_batch.shape[0]
                dz2 = y_pred - y_batch
                dW2 = np.dot(self.a1.T, dz2) / m
                db2 = np.sum(dz2, axis=0, keepdims=True) / m
                dz1 = np.dot(dz2, self.W2.T) * self.a1 * (1 - self.a1)
                dW1 = np.dot(X_batch.T, dz1) / m
                db1 = np.sum(dz1, axis=0, keepdims=True) / m
                self.W2 -= self.lr * dW2
                self.b2 -= self.lr * db2
                self.W1 -= self.lr * dW1
                self.b1 -= self.lr * db1
            train_pred = self.forward(X)
            loss = -np.mean(np.sum(y * np.log(train_pred + 1e-8), axis=1))
            self.losses.append(loss)
            if epoch % 5 == 0:
                print("Epoch:", epoch, "Loss:", loss)
    def predict(self, X):
        output = self.forward(X)
        return np.argmax(output, axis=1)



In [ ]:
iris = load_iris()

X = iris.data
y = iris.target
print("X shape:", X.shape)
print("y shape:", y.shape)
print("Classes:", iris.target_names)



In [ ]:
binary_filter = y < 2

X_binary = X[binary_filter]
y_binary = y[binary_filter]
print("X_binary shape:", X_binary.shape)
print("y_binary shape:", y_binary.shape)
print("Unique classes:", np.unique(y_binary))



In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X_binary,
    y_binary,
    test_size=0.2,
    random_state=42
)
print("Training :", X_train.shape[0])
print("Testing :", X_test.shape[0])



In [ ]:
scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)
print("X_train after scaling:")
print(X_train[:18])



In [ ]:
model = NeuralNetwork(
    input_size=X_train.shape[1],
    hidden_size=7,
    output_size=1,
    learning_rate=0.1
)

model.train(X_train, y_train, epochs=3000)



In [ ]:
y_pred = model.predict(X_test)
accuracy = accuracy_score(y_test, y_pred)

print("Accuracy:", accuracy)
print("\nConfusion Matrix:")
print(confusion_matrix(y_test, y_pred))
print("\nClassification Report:")
print(classification_report(y_test, y_pred))



In [ ]:
plt.figure(figsize=(10,7))
plt.plot(model.losses, color="#7FB3D5", linewidth=2)
plt.title("IRIS From Scratch")
plt.xlabel("Epoch")
plt.ylabel("Loss")
plt.grid(alpha=0.5)
plt.show()



In [ ]:
(X_train_mnist, y_train_mnist), (X_test_mnist, y_test_mnist) = mnist.load_data()

print("X_train shape:", X_train_mnist.shape)
print("y_train shape:", y_train_mnist.shape)
print("X_test shape:", X_test_mnist.shape)
print("y_test shape:", y_test_mnist.shape)



In [ ]:
X_train_mnist = X_train_mnist[:19700]
y_train_mnist = y_train_mnist[:19700]
X_test_mnist = X_test_mnist[:5700]
y_test_mnist = y_test_mnist[:5700]
X_train_mnist = X_train_mnist.reshape(X_train_mnist.shape[0], 28 * 28)
X_test_mnist = X_test_mnist.reshape(X_test_mnist.shape[0], 28 * 28)
X_train_mnist = X_train_mnist / 255.0
X_test_mnist = X_test_mnist / 255.0

print("X_train after reshape:", X_train_mnist.shape)
print("X_test after reshape:", X_test_mnist.shape)



In [ ]:
# One-hot encode the MNIST labels (missing step added so the notebook runs end-to-end)
y_train_onehot = np.eye(10)[y_train_mnist]
print("y_train_onehot shape:", y_train_onehot.shape)
print(y_train_onehot[:7])



In [ ]:
mnist_model = NeuralNetworkMNIST(
    input_size=784,
    hidden_size=64,
    output_size=10,
    learning_rate=0.5
)
mnist_model.train(X_train_mnist, y_train_onehot, epochs=100)



In [ ]:
y_pred_mnist = mnist_model.predict(X_test_mnist)
mnist_accuracy = accuracy_score(y_test_mnist, y_pred_mnist)

print("MNIST From Scratch Accuracy:", mnist_accuracy)
print("\nConfusion Matrix:")
print(confusion_matrix(y_test_mnist, y_pred_mnist))
print("\nClassification Report:")
print(classification_report(y_test_mnist, y_pred_mnist))



In [ ]:
plt.figure(figsize=(10,7))
plt.plot(mnist_model.losses, color="#A8D5BA", linewidth=2)
plt.title("MNIST From Scratch")
plt.xlabel("Epoch")
plt.ylabel("Loss")
plt.grid(alpha=0.5)
plt.show()



In [ ]:
keras_iris_model = Sequential()
keras_iris_model.add(Input(shape=(X_train.shape[1],)))
keras_iris_model.add(Dense(5, activation="sigmoid"))
keras_iris_model.add(Dense(1, activation="sigmoid"))
keras_iris_model.compile(
    optimizer="adam",
    loss="binary_crossentropy",
    metrics=["accuracy"]
)
iris_history = keras_iris_model.fit(
     X_train,
     y_train,
     epochs=320,
    verbose=0
)
iris_loss, iris_acc = keras_iris_model.evaluate(X_test, y_test, verbose=0)
print("Keras IRIS Accuracy:",iris_acc)
print("Keras IRIS Loss:",iris_loss)



In [ ]:
plt.figure(figsize=(10,7))
plt.plot(iris_history.history["loss"], color="#D7BDE2", linewidth=2)
plt.title("Keras IRIS - Loss Curve")
plt.xlabel("Epoch")
plt.ylabel("Loss")
plt.grid(alpha=0.5)
plt.show()



In [ ]:
(X_train_k, y_train_k), (X_test_k, y_test_k) = mnist.load_data()
X_train_k = X_train_k[:15200]
y_train_k = y_train_k[:15200]
X_test_k = X_test_k[:3400]
y_test_k = y_test_k[:3400]
X_train_k = X_train_k.reshape(X_train_k.shape[0], 28 * 28) / 255.0
X_test_k = X_test_k.reshape(X_test_k.shape[0], 28 * 28) / 255.0
y_train_k_cat = to_categorical(y_train_k, 10)
y_test_k_cat = to_categorical(y_test_k, 10)

print("X_train_k:", X_train_k.shape)
print("y_train_k_cat:", y_train_k_cat.shape)



In [ ]:
keras_mnist_model = Sequential()
keras_mnist_model.add(Input(shape=(784,)))
keras_mnist_model.add(Dense(64, activation="sigmoid"))
keras_mnist_model.add(Dense(10, activation="softmax"))
keras_mnist_model.compile(
    optimizer="adam",
    loss="categorical_crossentropy",
    metrics=["accuracy"]
)
mnist_history = keras_mnist_model.fit(
    X_train_k,
    y_train_k_cat,
    epochs=10,
    batch_size=32,
    verbose=1
)



In [ ]:
mnist_loss, mnist_acc = keras_mnist_model.evaluate(X_test_k, y_test_k_cat, verbose=0)

print("Keras MNIST Accuracy:", mnist_acc)
print("Keras MNIST Loss:", mnist_loss)



In [ ]:
plt.plot(mnist_history.history["loss"])
plt.title("Keras MNIST Multi-Class Classification")
plt.xlabel("Epoch")
plt.ylabel("Loss")
plt.show()



In [ ]:
comparison_df = pd.DataFrame({
    "Dataset": ["IRIS", "IRIS", "MNIST", "MNIST"],
    "Model": ["From Scratch", "Keras", "From Scratch", "Keras"],
    "Accuracy": [accuracy, iris_acc, mnist_accuracy, mnist_acc],
    "Loss": [model.losses[-1], iris_loss, mnist_model.losses[-1], mnist_loss]
})
comparison_df



In [ ]:
plt.style.use("seaborn-v0_8-whitegrid")



In [ ]:
plt.figure(figsize=(10,7))
labels = comparison_df["Dataset"] + " - " + comparison_df["Model"]
values = comparison_df["Accuracy"]
colors = ["#7FB3D5", "#D7BDE2", "#A8D5BA", "#F5CBA7"]
bars = plt.bar(labels, values, color=colors)
for i, v in enumerate(values):
    plt.text(i, v + 0.02, f"{v:.2f}", ha="center", fontsize=10)
plt.title("Accuracy Comparison")
plt.ylabel("Accuracy")
plt.ylim(0, 1.1)
plt.xticks(rotation=25)
plt.grid(axis="y", alpha=0.5)

plt.show()



In [ ]:
plt.figure(figsize=(10,7))
labels = comparison_df["Dataset"] + " - " + comparison_df["Model"]
values = comparison_df["Loss"]
bars = plt.bar(labels, values)
colors = ["#B8E0D2", "#AED6F1", "#FAD7A0", "#E8DAEF"]
for bar, color in zip(bars, colors):
    bar.set_color(color)
for i, v in enumerate(values):
    plt.text(i, v + 0.02, f"{v:.2f}", ha='center', fontsize=10)
plt.title("Loss Comparison", fontsize=14)
plt.ylabel("Loss")
plt.xticks(rotation=25)

plt.show()



In [ ]:
plt.figure(figsize=(10, 7))
plt.plot(model.losses, label="IRIS", linewidth=2)
plt.plot(mnist_model.losses, label="MNIST", linewidth=2)
plt.title("Training Loss Curves")
plt.xlabel("Epoch")
plt.ylabel("Loss")
plt.legend()
plt.show()



1. Introduction
This project implements a Neural Network from scratch using NumPy and compares its performance with Keras-based implementations. The models are evaluated on two datasets: IRIS (binary classification) and MNIST (multi-class classification), in order to analyze differences in performance, complexity, and usability
2. Datasets
IRIS Dataset: Used for binary classification by selecting two classes only.
MNIST Dataset: A multi-class classification problem consisting of handwritten digit images (10 classes)
3. From Scratch Implementation

A neural network with one hidden layer was implemented manually using NumPy. The implementation includes:

Forward propagation
Backpropagation
Loss computation (Binary Cross-Entropy for IRIS and Categorical Cross-Entropy for MNIST)

This approach provides full control over the learning process and helps in understanding the internal mechanics of neural networks

4. Results

The performance of both the from-scratch and Keras implementations was evaluated on the IRIS and MNIST datasets.

IRIS Dataset:

The model implemented from scratch achieved perfect performance with an accuracy of 1.00. The Keras model also achieved an accuracy of 1.00, indicating that both approaches are highly effective for simple binary classification tasks.

MNIST Dataset:

The from-scratch implementation achieved a high accuracy of approximately 0.96, demonstrating strong capability in handling multi-class classification problems. The Keras model achieved an accuracy of approximately 0.93, which is also strong but slightly lower than the from-scratch model in this specific setup.

Overall, both approaches performed well. The from-scratch model showed slightly better performance in this experiment on the MNIST dataset, but both models achieved strong and comparable results

5. Comparison

The comparison between the from-scratch implementation and the Keras-based models highlights key differences in performance, complexity, and usability.

The from-scratch neural network demonstrated strong performance across both datasets. It achieved perfect accuracy on the IRIS dataset and high accuracy on MNIST. However, this approach required manual implementation of all components, including forward propagation, backpropagation, and loss computation. As a result, the development process was more complex, time-consuming, and sensitive to hyperparameter tuning.

On the other hand, Keras provided a more efficient and user-friendly solution. The models were easier to build, required significantly less code, and benefited from optimized internal implementations. Training was faster and more stable, making Keras more suitable for real-world applications and large-scale problems

Advantages & Limitations

Advantages of From Scratch:

Full control over implementation
Deeper understanding of neural network operations

Limitations:

Slower training
More complex debugging and tuning

Advantages of Keras:

Faster development
Optimized and reliable performance
Easier experimentation with different architectures
